In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# ==================================================
# 1. CẤU HÌNH
# ==================================================

SEED = 42
TOP_K = 50

CSV_PATH = "/kaggle/input/datasets/bondyyyyy/near-dup/near_duplicates.csv"
IMAGE_DIR = None

# Chỉ áp dụng khi có danh sách từ S0 chính thức
S0_VALID_IDS = None
S0_EXCLUDED_IDS = set()

WEIGHTS = {
    "ssim": 0.25,
    "ncc": 0.25,
    "phash_distance": 0.20,
    "dhash_distance": 0.20,
    "mask_iou": 0.10
}

HIGHER_BETTER = {
    "ssim", "ncc", "mask_iou"
}

# ==================================================
# 2. TÌM FILE CSV VÀ THƯ MỤC ẢNH
# ==================================================

if CSV_PATH is None:

    matches = sorted([
        p for p in Path("/kaggle/input").rglob("*.csv")
        if p.name.lower() in {
            "near_duplicate.csv",
            "near_duplicates.csv"
        }
    ])

    if not matches:
        raise FileNotFoundError(
            "Không tìm thấy near_duplicates.csv. "
            "Hãy sửa CSV_PATH."
        )

    CSV_PATH = next(
        (
            p for p in matches
            if p.name.lower() == "near_duplicates.csv"
        ),
        matches[0]
    )

CSV_PATH = Path(CSV_PATH)

if IMAGE_DIR is None:

    candidates = [
        Path(
            "/kaggle/input/competitions/"
            "aptos2019-blindness-detection/train_images"
        ),
        Path(
            "/kaggle/input/"
            "aptos2019-blindness-detection/train_images"
        )
    ]

    IMAGE_DIR = next(
        (p for p in candidates if p.is_dir()),
        None
    )

    if IMAGE_DIR is None:

        folders = [
            p for p in Path("/kaggle/input").rglob("train_images")
            if p.is_dir() and "aptos" in str(p).lower()
        ]

        IMAGE_DIR = folders[0] if folders else None

if IMAGE_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy train_images. "
        "Hãy sửa IMAGE_DIR."
    )

IMAGE_DIR = Path(IMAGE_DIR)

print("CSV:", CSV_PATH)
print("IMAGE_DIR:", IMAGE_DIR)

# ==================================================
# 3. ĐỌC DỮ LIỆU
# ==================================================

available_columns = pd.read_csv(
    CSV_PATH, nrows=0
).columns.tolist()

columns_needed = [
    "id_1", "id_2",
    "grade_1", "grade_2",
    "ssim", "ncc", "mask_iou",
    "phash_distance", "dhash_distance",
    "same_exact_cluster",
    "pixel_sha256_1", "pixel_sha256_2",
    "comparison_usable_1",
    "comparison_usable_2",
    "candidate_category",
    "high_similarity",
    "decision"
]

usecols = [
    c for c in columns_needed
    if c in available_columns
]

required = ["id_1", "id_2"] + list(WEIGHTS)

missing = [
    c for c in required
    if c not in available_columns
]

if missing:
    raise ValueError(
        f"Thiếu các cột bắt buộc: {missing}"
    )

df = pd.read_csv(
    CSV_PATH,
    usecols=usecols,
    low_memory=False
)

df["id_1"] = df["id_1"].astype(str).str.strip()
df["id_2"] = df["id_2"].astype(str).str.strip()

original_count = len(df)

print("\nSố cặp ban đầu:", original_count)

# ==================================================
# 4. LOẠI EXACT DUPLICATES THEO S0
# ==================================================

def is_true(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

exact = pd.Series(False, index=df.index)

if "same_exact_cluster" in df:
    exact |= is_true(df["same_exact_cluster"])

if "candidate_category" in df:
    exact |= df["candidate_category"].eq("exact")

if "decision" in df:
    exact |= df["decision"].eq("merge_exact")

if {
    "pixel_sha256_1",
    "pixel_sha256_2"
}.issubset(df.columns):

    exact |= (
        df["pixel_sha256_1"].notna()
        & df["pixel_sha256_2"].notna()
        & df["pixel_sha256_1"].eq(df["pixel_sha256_2"])
    )

print("Exact bị loại:", int(exact.sum()))

df = df.loc[~exact].copy()

# ==================================================
# 5. LỌC ỨNG VIÊN CÓ THỂ SO SÁNH
# ==================================================

for col in [
    "comparison_usable_1",
    "comparison_usable_2"
]:
    if col in df.columns:
        df = df.loc[is_true(df[col])].copy()

# Không tự suy diễn tập hợp lệ của S0
if S0_VALID_IDS is not None:

    valid = set(map(str, S0_VALID_IDS))

    df = df.loc[
        df["id_1"].isin(valid)
        & df["id_2"].isin(valid)
    ].copy()

if S0_EXCLUDED_IDS:

    excluded = set(map(str, S0_EXCLUDED_IDS))

    df = df.loc[
        ~df["id_1"].isin(excluded)
        & ~df["id_2"].isin(excluded)
    ].copy()

# ==================================================
# 6. KHỬ TRÙNG CẶP A-B VÀ B-A
# ==================================================

df = df.loc[df["id_1"] != df["id_2"]].copy()

df["_pair_key"] = (
    np.minimum(df["id_1"], df["id_2"])
    + "|"
    + np.maximum(df["id_1"], df["id_2"])
)

df = df.drop_duplicates(
    subset="_pair_key",
    keep="first"
).copy()

print("Ứng viên cần xếp hạng:", len(df))

if len(df) < TOP_K:
    raise ValueError("Không đủ 50 cặp ứng viên.")

# ==================================================
# 7. CHUYỂN CÁC METRIC SANG THỨ HẠNG PHÂN VỊ
# ==================================================

N = len(df)

ranks = {}

for metric in WEIGHTS:

    values = pd.to_numeric(
        df[metric],
        errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

    df[metric] = values

    # Nếu giá trị gốc càng cao càng giống:
    # phân vị cao là tốt.
    #
    # Với hash distance:
    # phân vị cao tương ứng khoảng cách thấp.
    ranks[metric] = values.rank(
        pct=True,
        ascending=(metric in HIGHER_BETTER),
        method="average"
    )

R = pd.DataFrame(ranks, index=df.index)

# Không thưởng cho metric bị thiếu
missing_metric_count = R.isna().sum(axis=1)

# Missing -> mức phân vị thấp nhất
R = R.fillna(1.0 / (N + 1))

# ==================================================
# 8. TÍNH ĐIỂM TỔNG HỢP
# ==================================================

# Weighted geometric mean:
#
# score = 100 * exp(
#     sum(weight_i * log(percentile_i))
# )
#
# Không phải xác suất trùng ảnh.
# Một metric quá yếu sẽ làm giảm điểm tổng hợp.

weighted_log_score = np.zeros(N, dtype=float)

for metric, weight in WEIGHTS.items():

    weighted_log_score += (
        weight * np.log(
            R[metric].clip(lower=1.0 / (N + 1))
        ).to_numpy()
    )

df["similarity_priority"] = (
    100.0 * np.exp(weighted_log_score)
)

df["missing_metric_count"] = missing_metric_count

# Độ bất đồng giữa các metric để tham khảo
df["metric_disagreement"] = R.std(
    axis=1,
    ddof=0
)

# ==================================================
# 9. XẾP HẠNG TOÀN BỘ TỪ CAO XUỐNG THẤP
# ==================================================

RANKED = df.sort_values(
    [
        "similarity_priority",
        "ssim",
        "ncc",
        "id_1",
        "id_2"
    ],
    ascending=[
        False, False, False, True, True
    ],
    kind="mergesort"
).reset_index(drop=True)

RANKED["rank"] = np.arange(
    1, len(RANKED) + 1
)

# Lấy 50 cặp đầu tiên
TOP50 = RANKED.head(TOP_K).copy()

# ==================================================
# 10. KIỂM TRA KẾT QUẢ
# ==================================================

print("\n" + "=" * 65)
print("KẾT QUẢ XẾP HẠNG NEAR-DUPLICATE")
print("=" * 65)

print("Tổng ứng viên:", len(RANKED))
print("Top cần kiểm tra:", len(TOP50))
print("Sử dụng 5 metric:", list(WEIGHTS))

if "high_similarity" in TOP50.columns:

    n_high = int(
        is_true(TOP50["high_similarity"]).sum()
    )

    print("High-similarity trong Top 50:", n_high)

print("\nTOP 50 CẶP:")

display_columns = [
    "rank",
    "id_1", "id_2",
    "grade_1", "grade_2",
    "similarity_priority",
    "ssim", "ncc", "mask_iou",
    "phash_distance", "dhash_distance",
    "metric_disagreement"
]

display(
    TOP50[
        [c for c in display_columns if c in TOP50.columns]
    ].round(5)
)

print("\nXếp hạng hoàn tất.")
print("RANKED chứa toàn bộ thứ hạng trong RAM.")
print("TOP50 chứa 50 cặp đứng đầu.")
print("Không có file nào được tạo.")



In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from pathlib import Path

# ==================================================
# 1. CẤU HÌNH HIỂN THỊ
# ==================================================

START_RANK = 501
END_RANK = 550

ROWS_PER_FIG = 5

# Ảnh lớn để quan sát võng mạc
IMAGE_SIZE = 1400

FIG_WIDTH = 22
HEIGHT_PER_PAIR = 7.2
DPI = 100

CLASS_NAMES = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR"
}

# ==================================================
# 2. HÀM ĐỌC ẢNH
# ==================================================

def load_review_image(image_id):

    image_id = str(image_id)

    for ext in [".png", ".jpg", ".jpeg"]:

        path = IMAGE_DIR / f"{image_id}{ext}"

        if not path.is_file():
            continue

        with Image.open(path) as im:

            img = im.convert("RGB")

            resampling = getattr(
                Image, "Resampling", Image
            ).LANCZOS

            img.thumbnail(
                (IMAGE_SIZE, IMAGE_SIZE),
                resampling
            )

            return img.copy()

    return None

# ==================================================
# 3. ĐỊNH DẠNG GRADE
# ==================================================

def grade_text(value):

    if pd.isna(value):
        return "Grade: Unknown"

    try:
        grade = int(value)
    except (ValueError, TypeError):
        return "Grade: Unknown"

    name = CLASS_NAMES.get(grade, "Unknown")

    return f"Grade {grade} - {name}"

# ==================================================
# 4. ĐỊNH DẠNG METRIC
# ==================================================

def metrics_text(row):

    return (
        f"SSIM: {row['ssim']:.4f}   |   "
        f"NCC: {row['ncc']:.4f}   |   "
        f"Mask IoU: {row['mask_iou']:.4f}   |   "
        f"pHash: {row['phash_distance']:.0f}   |   "
        f"dHash: {row['dhash_distance']:.0f}"
    )

# ==================================================
# 5. HIỂN THỊ THEO THỨ HẠNG
# ==================================================

start_rank = max(1, int(START_RANK))
end_rank = min(int(END_RANK), len(RANKED))

if start_rank > end_rank:
    raise ValueError("Khoảng thứ hạng không hợp lệ.")

for batch_start in range(
    start_rank - 1,
    end_rank,
    ROWS_PER_FIG
):

    batch = RANKED.iloc[
        batch_start:min(
            batch_start + ROWS_PER_FIG,
            end_rank
        )
    ]

    n = len(batch)

    fig = plt.figure(
        figsize=(
            FIG_WIDTH,
            HEIGHT_PER_PAIR * n
        ),
        dpi=DPI
    )

    # 1 hàng tiêu đề + 1 hàng ảnh cho mỗi cặp
    gs = fig.add_gridspec(
        nrows=2 * n,
        ncols=2,
        height_ratios=[0.16, 1.0] * n,
        left=0.035,
        right=0.965,
        top=0.987,
        bottom=0.025,
        hspace=0.20,
        wspace=0.05
    )

    for i, (_, row) in enumerate(batch.iterrows()):

        rank = int(row["rank"])

        id_a = str(row["id_1"])
        id_b = str(row["id_2"])

        grade_a = row.get("grade_1", np.nan)
        grade_b = row.get("grade_2", np.nan)

        score = float(row["similarity_priority"])

        # ----------------------------------
        # TIÊU ĐỀ VÀ METRIC
        # ----------------------------------

        info_ax = fig.add_subplot(
            gs[2 * i, :]
        )

        info_ax.axis("off")

        info_ax.text(
            0.0,
            0.80,
            (
                f"HẠNG {rank:03d}/{len(RANKED)}"
                f"   |   ĐIỂM ƯU TIÊN: {score:.4f}/100"
            ),
            transform=info_ax.transAxes,
            fontsize=17,
            fontweight="bold",
            va="top"
        )

        info_ax.text(
            0.0,
            0.10,
            metrics_text(row),
            transform=info_ax.transAxes,
            fontsize=13,
            va="bottom"
        )

        # ----------------------------------
        # HIỂN THỊ 2 ẢNH CẠNH NHAU
        # ----------------------------------

        pair_images = [
            ("A", id_a, grade_a),
            ("B", id_b, grade_b)
        ]

        for j, (side, image_id, grade) in enumerate(
            pair_images
        ):

            ax = fig.add_subplot(
                gs[2 * i + 1, j]
            )

            img = load_review_image(image_id)

            if img is not None:

                ax.imshow(
                    img,
                    interpolation="nearest"
                )

            else:

                ax.text(
                    0.5,
                    0.5,
                    f"IMAGE NOT FOUND\n{image_id}",
                    transform=ax.transAxes,
                    ha="center",
                    va="center",
                    fontsize=15
                )

            ax.set_title(
                f"ẢNH {side}: {image_id}"
                f"  |  {grade_text(grade)}",
                fontsize=15,
                fontweight="bold",
                pad=10
            )

            ax.axis("off")

    plt.show()
    plt.close(fig)

print("=" * 60)
print(
    f"ĐÃ HIỂN THỊ THỨ HẠNG "
    f"{start_rank} ĐẾN {end_rank}"
)
print("=" * 60)
print("Không tạo file.")
print("Không xác nhận near-duplicate tự động.")
print("Không gộp cụm hoặc thay đổi S0.")
